In [36]:
import os
import glob
import re
import unicodedata

import numpy as np
import pandas as pd

BASE   = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW    = os.path.join(BASE, "Datos_originales")
MOD    = os.path.join(BASE, "Datos_modificados")
CLEAN  = os.path.join(BASE, "Datos_limpios")   # salida de este script
os.makedirs(CLEAN, exist_ok=True)

In [37]:
def resumen(df: pd.DataFrame, nombre: str):
    """Imprime métricas de calidad básicas (DAMA-DMBOK)."""
    total = df.shape[0] * df.shape[1]
    nulos = df.isnull().sum().sum()
    dupes = df.duplicated().sum()
    print(f"  {nombre}")
    print(f"  Filas        : {df.shape[0]:>8,}")
    print(f"  Columnas     : {df.shape[1]:>8}")
    print(f"  Nulos        : {nulos:>8,}  ({nulos/total*100:.1f} %)")
    print(f"  Duplicados   : {dupes:>8,}  ({dupes/df.shape[0]*100:.1f} %)")
    cols_nulas = df.columns[df.isnull().any()].tolist()
    if cols_nulas:
        pct = (df[cols_nulas].isnull().mean() * 100).round(1)
        print(f"\n  Columnas con nulos:")
        for col, p in pct.items():
            print(f"    {col:<40} {p:>5.1f} %")


def normalizar_nombre(texto: str):
    """
    Normalización canónica de nombres propios para record linkage:
      - strip de espacios extremos
      - mayúsculas
      - elimina acentos / diacríticos
      - colapsa espacios múltiples
      - elimina caracteres no alfanuméricos (salvo espacio)
    """
    if pd.isna(texto):
        return np.nan
    texto = str(texto).strip().upper()
    # Descompone caracteres Unicode y descarta marcas de acento
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    # Elimina todo excepto letras, dígitos y espacios
    texto = re.sub(r"[^A-Z0-9 ]", " ", texto)
    # Colapsa espacios
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def normalizar_equipo(nombre: str):
    """
    Normalización de nombres de clubes / equipos.
    Aplica normalizar_nombre y además:
      - Elimina palabras genéricas comunes (F.C., FC, C.F., S.A., etc.)
      - Unifica variantes conocidas de equipos de Liga MX
    """
    if pd.isna(nombre):
        return np.nan
    s = normalizar_nombre(nombre)

    # Eliminar sufijos/prefijos genéricos
    for pat in [r"\bF\.?C\.?\b", r"\bC\.?F\.?\b", r"\bA\.?C\.?\b",
                r"\bS\.?A\.?\b", r"\bDE\b", r"\bLOS\b", r"\bLAS\b",
                r"\bEL\b", r"\bLA\b"]:
        s = re.sub(pat, "", s)
    s = re.sub(r"\s+", " ", s).strip()

    # Diccionario de variantes conocidas → nombre canónico
    ALIAS_EQUIPOS: dict[str, str] = {
        # Cruz Azul
        "CRUZ AZUL": "CRUZ AZUL",
        "LA MAQUINA": "CRUZ AZUL",
        "LA MAQUINA CELESTE": "CRUZ AZUL",
        # América
        "AMERICA": "CLUB AMERICA",
        "AGUILAS": "CLUB AMERICA",
        "AGUILAS AMERICA": "CLUB AMERICA",
        # Chivas
        "GUADALAJARA": "CHIVAS GUADALAJARA",
        "CHIVAS": "CHIVAS GUADALAJARA",
        "DEPORTIVO GUADALAJARA": "CHIVAS GUADALAJARA",
        # Tigres
        "TIGRES": "UANL TIGRES",
        "UANL": "UANL TIGRES",
        # Monterrey
        "MONTERREY": "MONTERREY",
        "RAYADOS": "MONTERREY",
        # Pumas
        "PUMAS": "PUMAS UNAM",
        "UNAM": "PUMAS UNAM",
        # Atlas
        "ATLAS": "ATLAS GUADALAJARA",
        # Necaxa
        "NECAXA": "NECAXA",
        "RAYOS": "NECAXA",
        # Santos
        "SANTOS": "SANTOS LAGUNA",
        "SANTOS LAGUNA": "SANTOS LAGUNA",
        # Toluca
        "TOLUCA": "TOLUCA",
        "DIABLOS ROJOS": "TOLUCA",
        # Pachuca
        "PACHUCA": "PACHUCA",
        "TUZOS": "PACHUCA",
    }
    return ALIAS_EQUIPOS.get(s, s)

def normalizar_posicion(valor: str):
    """Traduce abreviatura FIFA a nombre canónico Transfermarkt.
    Si el valor ya es un nombre largo (viene de TM), lo devuelve sin cambios."""
    if pd.isna(valor):
        return np.nan
    s = str(valor).strip().upper()
    EQUIV_POSICIONES: dict[str, str] = {
        # FIFA 23 → Transfermarkt
        "GK"    : "Goalkeeper",
        "SW"    : "Sweeper",
        "RB"    : "Right-Back",
        "LB"    : "Left-Back",
        "CB"    : "Centre-Back",
        "RWB"   : "Right Wing Back",
        "LWB"   : "Left Wing Back",
        "CDM"   : "Defensive Midfield",
        "CM"    : "Central Midfield",
        "RM"    : "Right Midfield",
        "LM"    : "Left Midfield",
        "CAM"   : "Attacking Midfield",
        "RW"    : "Right Winger",
        "LW"    : "Left Winger",
        "CF"    : "Centre-Forward",
        "ST"    : "Centre-Forward",
        "RF"    : "Right Winger",
        "LF"    : "Left Winger",
    }
    return EQUIV_POSICIONES.get(s, valor)  # si no está en el dict, respeta el original

def exportar(df, directorio, nombre_archivo):
    os.makedirs(directorio, exist_ok=True)
    ruta = os.path.join(directorio, nombre_archivo)
    df.to_parquet(ruta, index=False)
    print(f"\nExportado → {ruta}  ({len(df):,} filas)")

# 1. LigaMX 2016-2024  (JSON)

In [38]:
def limpiar_ligamx():
    """
    Limpieza de partidos LigaMX 2016-2024.

    Problemas identificados en el perfilado:
      - 22.4 % de celdas nulas  (venue_id, venue_city, home_win, away_win,
        goles de tiempo extra y penales)
      - Columna 'timezone' constante → irrelevante para RL
      - Tipos: 'date' llega como string, goles extra_time como Categorical
        en lugar de numérico
      - Inconsistencia en nombres de equipos entre temporadas (Problemática #4)
      - Columnas de goles duplicadas: home_goals vs home_goals_fulltime

    Acciones:
      1. Parsear 'date' como datetime y extraer date_key (YYYY-MM-DD) para RL
      2. Convertir columnas de goles extra_time a numérico (int, relleno 0)
      3. Normalizar home_team / away_team con normalizar_equipo()
      4. Crear columna name_key_home / name_key_away (para bloqueo en RL)
      5. Eliminar columna constante 'timezone'
      6. Eliminar columnas de goles redundantes (fulltime = duplicado de goals)
      7. Imputar venue_city con 'DESCONOCIDA' (categórica, no bloqueante)
      8. Dejar nulos en referee (irrelevante para RL)
    """

    ruta = os.path.join(MOD, "LigaMX", "2016-2024_liga_mx.json")
    df = pd.read_json(ruta)
    resumen(df, "ANTES — LigaMX")

    # 1. Fecha
    df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
    df["date_key"] = df["date"].dt.strftime("%Y-%m-%d")   # clave para RL
    df["season"] = df["season"].astype("Int64")

    # 2. Goles extra_time → numérico entero (NaN → 0, solo si hay penales)
    for col in ["home_goals_extra_time", "away_goals_extratime",
                "home_goals_penalty", "away_goals_penalty"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    # 3. Normalizar equipos
    df["home_team_norm"] = df["home_team"].apply(normalizar_equipo)
    df["away_team_norm"] = df["away_team"].apply(normalizar_equipo)

    # 4. Eliminar columna constante y redundantes
    cols_drop = ["timezone"]
    # fulltime == goals cuando no hay prórroga → verificar y eliminar si iguales
    for par in [("home_goals", "home_goals_fulltime"),
                ("away_goals", "away_goals_fulltime")]:
        c1, c2 = par
        if c1 in df.columns and c2 in df.columns:
            if df[c1].equals(df[c2]):
                cols_drop.append(c2)
    df.drop(columns=[c for c in cols_drop if c in df.columns], inplace=True)

    # 5. Imputar venue_city
    df["venue_city"] = df["venue_city"].fillna("DESCONOCIDA")

    # 6. Reset index
    df.reset_index(drop=True, inplace=True)

    resumen(df, "DESPUÉS — LigaMX")
    exportar(df, os.path.join(CLEAN, "LigaMX"), "ligamx_clean.csv")
    return df

# 2. Transfermarkt players  (XLSX)

In [39]:
def limpiar_transfermarkt():
    """
    Limpieza de jugadores Transfermarkt.

    Problemas identificados en el perfilado:
      - 14.8 % de celdas nulas: first_name, city_of_birth, agent_name,
        contract_expiration_date, market_value_in_eur, int. caps/goals,
        foot, height_in_cm
      - 'date_of_birth' / 'contract_expiration_date' como strings
      - Nombres con caracteres especiales / diacríticos → inconsistencia en RL
      - player_code y url son URLs/slugs, no normalizados
      - Columnas image_url / url irrelevantes para RL

    Acciones:
      1. Parsear fechas (date_of_birth, contract_expiration_date)
      2. Crear name_key = normalizar_nombre(name)  [columna bloqueante RL]
      3. Crear dob_key  = fecha YYYY-MM-DD          [columna bloqueante RL]
      4. Imputar foot / height_in_cm con moda / mediana de posición
      5. Imputar market_value_in_eur con mediana de sub_position
      6. Eliminar columnas no útiles para RL (image_url, player_code slug)
      7. Estandarizar country_of_citizenship → mayúsculas sin acento
    """
    ruta = os.path.join(MOD, "FootballDatafromTransfermarkt", "players.xlsx")
    df = pd.read_excel(ruta, engine="openpyxl")
    resumen(df, "ANTES — Transfermarkt")

    # 1. Fechas
    for col_fecha in ["date_of_birth", "contract_expiration_date"]:
        if col_fecha in df.columns:
            df[col_fecha] = pd.to_datetime(df[col_fecha], errors="coerce")

    # 2. Claves para RL
    df["name_key"] = df["name"].apply(normalizar_nombre)
    df["first_name_key"] = df["first_name"].apply(normalizar_nombre)
    df["last_name_key"]  = df["last_name"].apply(normalizar_nombre)
    df["dob_key"] = df["date_of_birth"].dt.strftime("%Y-%m-%d")

    # 3. Normalizar country_of_citizenship
    for col in ["country_of_citizenship", "country_of_birth"]:
        if col in df.columns:
            df[col] = df[col].apply(normalizar_nombre)

    # 4. Imputar foot con moda por sub_position
    if "foot" in df.columns and "sub_position" in df.columns:
        moda_foot = (
            df.dropna(subset=["foot"])
              .groupby("sub_position")["foot"]
              .agg(lambda x: x.mode().iloc[0] if len(x.mode()) else np.nan)
        )
        def _impute_foot(row):
            if pd.isna(row["foot"]) and row["sub_position"] in moda_foot.index:
                return moda_foot[row["sub_position"]]
            return row["foot"]
        df["foot"] = df.apply(_impute_foot, axis=1)

    # 5. Imputar height_in_cm con mediana por sub_position
    if "height_in_cm" in df.columns and "sub_position" in df.columns:
        mediana_altura = (
            df.dropna(subset=["height_in_cm"])
              .groupby("sub_position")["height_in_cm"]
              .median()
        )
        def _impute_height(row):
            if pd.isna(row["height_in_cm"]) and row["sub_position"] in mediana_altura.index:
                return mediana_altura[row["sub_position"]]
            return row["height_in_cm"]
        df["height_in_cm"] = df.apply(_impute_height, axis=1)

    # 6. Imputar market_value_in_eur con mediana por sub_position
    if "market_value_in_eur" in df.columns and "sub_position" in df.columns:
        mediana_valor = (
            df.dropna(subset=["market_value_in_eur"])
              .groupby("sub_position")["market_value_in_eur"]
              .median()
        )
        def _impute_valor(row):
            if pd.isna(row["market_value_in_eur"]) and row["sub_position"] in mediana_valor.index:
                return mediana_valor[row["sub_position"]]
            return row["market_value_in_eur"]
        df["market_value_in_eur"] = df.apply(_impute_valor, axis=1)

    # 7. Eliminar columnas no relevantes para RL
    cols_drop = ["image_url", "player_code", "agent_name",
                 "current_national_team_id", "current_club_domestic_competition_id"]
    df.drop(columns=[c for c in cols_drop if c in df.columns], inplace=True)

    # Normalizar posiciones al vocabulario canónico
    for col in ["position", "sub_position"]:
        if col in df.columns:
            df[col] = df[col].apply(normalizar_posicion)

    df.reset_index(drop=True, inplace=True)
    resumen(df, "DESPUÉS — Transfermarkt")
    exportar(df, os.path.join(CLEAN, "Transfermarkt"), "transfermarkt_players_clean.csv")
    return df

# 3. FIFA 23 male_coaches  (TXT / TSV)

In [40]:
def limpiar_fifa_coaches() -> pd.DataFrame:
    """
    Limpieza de entrenadores FIFA 23.

    Problemas identificados en el perfilado:
      - 9.6 % de celdas nulas: dob, face_url
      - coach_url es path relativo (no URL completa)
      - short_name con formato 'I. Apellido' → ambiguo para RL
      - long_name es la columna más útil para RL (99.5 % único)
      - nationality_name tiene solo 13 valores únicos pero con posible
        inconsistencia de escritura

    Acciones:
      1. Crear name_key = normalizar_nombre(long_name)   [bloqueante RL]
      2. Crear dob_key  = YYYY-MM-DD de dob
      3. Estandarizar nationality_name → normalizar_nombre()
      4. Construir coach_url_full = 'https://www.ea.com' + coach_url
      5. Eliminar face_url (irrelevante para RL)
      6. Separar long_name en first_name_key / last_name_key aproximados
    """

    ruta = os.path.join(MOD, "FIFA23complete_player_dataset", "male_coaches.txt")
    df = pd.read_csv(ruta, sep="\t", low_memory=False)
    resumen(df, "ANTES — FIFA23 coaches")

    # 1. Clave de nombre
    df["name_key"] = df["long_name"].apply(normalizar_nombre)

    # 2. Separar nombre en partes (aproximación: último token = apellido)
    def _split_name(nombre_key):
        if pd.isna(nombre_key):
            return np.nan, np.nan
        partes = str(nombre_key).split()
        if len(partes) == 1:
            return partes[0], np.nan
        return " ".join(partes[:-1]), partes[-1]

    df[["first_name_key", "last_name_key"]] = df["name_key"].apply(
        lambda x: pd.Series(_split_name(x))
    )

    # 3. Fecha de nacimiento
    if "dob" in df.columns:
        df["dob"] = pd.to_datetime(df["dob"], errors="coerce")
        df["dob_key"] = df["dob"].dt.strftime("%Y-%m-%d")
    else:
        df["dob_key"] = np.nan

    # 4. Normalizar nacionalidad
    if "nationality_name" in df.columns:
        df["nationality_name"] = df["nationality_name"].apply(normalizar_nombre)

    # 5. URL completa (opcional, útil si se cruza con otra fuente FIFA)
    if "coach_url" in df.columns:
        df["coach_url_full"] = "https://www.ea.com" + df["coach_url"].astype(str)

    # 6. Eliminar columnas irrelevantes para RL
    cols_drop = ["face_url", "coach_url"]
    df.drop(columns=[c for c in cols_drop if c in df.columns], inplace=True)

    df.reset_index(drop=True, inplace=True)
    resumen(df, "DESPUÉS — FIFA23 coaches")
    exportar(df, os.path.join(CLEAN, "FIFA23"), "fifa23_coaches_clean.csv")
    return df


# 4. OpenPublicDomain (CSV multi-temporada)

In [41]:
def limpiar_opfd() -> pd.DataFrame:
    """
    Limpieza de partidos históricos OpenPublicDomainFootballData.

    Problemas identificados en el perfilado:
      - 24.2 % de celdas nulas: Time, Timezone, HT, FT, UTC, ET, P, Comments
      - Columna 'Round' marcada como Unsupported/Rejected por YData
      - Nombres de equipos distintos a LigaMX: 'Atlas Guadalajara' vs 'Atlas'
        (Problemática #4)
      - FT y HT en formato '2-1' (string) → separar en goles_local / goles_visitante
      - Stage: 'Apertura' / 'Clausura' — ya consistente
      - Timezone: mezcla CDT/-0500 y otras zonas

    Acciones:
      1. Parsear Date + Time → datetime_key (YYYY-MM-DD) para RL
      2. Separar FT en ft_home_goals / ft_away_goals (int)
      3. Separar HT en ht_home_goals / ht_away_goals (int, puede ser NaN)
      4. Normalizar Team 1 / Team 2 con normalizar_equipo()
      5. Eliminar columnas poco densas o irrelevantes: ET, P, Comments, Timezone, UTC
      6. Renombrar columnas a snake_case para consistencia
    """

    archivos = sorted(glob.glob(
        os.path.join(RAW, "OpenPublicDomainFootballData", "*", "mx.1.csv")
    ))
    if not archivos:
        raise FileNotFoundError(
            f"No se encontraron archivos mx.1.csv bajo {RAW}/OpenPublicDomainFootballData/*/mx.1.csv"
        )
    df = pd.concat([pd.read_csv(f) for f in archivos], ignore_index=True)
    print(f"  Archivos cargados  : {len(archivos)} temporadas")
    resumen(df, "ANTES — OpenPublicDomain")

    # 1. Renombrar a snake_case
    rename_map = {
        "Stage": "stage",
        "Round": "round",
        "Date": "date",
        "Time": "time",
        "Timezone": "timezone",
        "Team 1": "home_team",
        "FT": "ft",
        "HT": "ht",
        "Team 2": "away_team",
        "UTC": "utc",
        "ET": "et",
        "P": "penalties",
        "Comments": "comments",
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

    # 2. Fecha
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce", dayfirst=False)
        df["date_key"] = df["date"].dt.strftime("%Y-%m-%d")

    # 3. Separar FT y HT
    def _split_score(score_str):
        """'2-1' → (2, 1);  NaN → (NaN, NaN)"""
        if pd.isna(score_str):
            return np.nan, np.nan
        partes = str(score_str).strip().split("-")
        if len(partes) != 2:
            return np.nan, np.nan
        try:
            return int(partes[0]), int(partes[1])
        except ValueError:
            return np.nan, np.nan

    if "ft" in df.columns:
        df[["ft_home_goals", "ft_away_goals"]] = df["ft"].apply(
            lambda x: pd.Series(_split_score(x))
        )
    if "ht" in df.columns:
        df[["ht_home_goals", "ht_away_goals"]] = df["ht"].apply(
            lambda x: pd.Series(_split_score(x))
        )

    # 4. Normalizar nombres de equipos
    for col in ["home_team", "away_team"]:
        if col in df.columns:
            df[f"{col}_norm"] = df[col].apply(normalizar_equipo)

    # 5. Eliminar columnas con alta tasa de nulos y sin valor para RL
    cols_drop = ["timezone", "utc", "et", "penalties", "comments",
                 "ft", "ht", "time", "round"]  # round = Unsupported
    df.drop(columns=[c for c in cols_drop if c in df.columns], inplace=True)

    df.reset_index(drop=True, inplace=True)
    resumen(df, "DESPUÉS — OpenPublicDomain")
    exportar(df, os.path.join(CLEAN, "OpenPublicDomain"), "opfd_clean.csv")
    return df

In [42]:
print("  LIMPIEZA PRE-RECORD LINKAGE — Cruz Azul Intelligence")

df_ligamx = limpiar_ligamx()
df_tm     = limpiar_transfermarkt()
df_fifa   = limpiar_fifa_coaches()
df_opfd   = limpiar_opfd()

print("\n\n" + "═"*55)
print("  RESUMEN FINAL DE DATASETS LIMPIOS")
print("═"*55)
resúmenes = {
    "LigaMX"           : df_ligamx,
    "Transfermarkt"    : df_tm,
    "FIFA23 coaches"   : df_fifa,
    "OpenPublicDomain" : df_opfd,
}
for nombre, df in resúmenes.items():
    total = df.shape[0] * df.shape[1]
    nulos = df.isnull().sum().sum()
    print(f"\n  {nombre:<22}  {df.shape[0]:>7,} filas × {df.shape[1]:>2} cols  "
            f"| nulos: {nulos/total*100:.1f} %")

print("\nTodos los datasets limpios exportados a:", CLEAN)


  LIMPIEZA PRE-RECORD LINKAGE — Cruz Azul Intelligence
  ANTES — LigaMX
  Filas        :    2,876
  Columnas     :       23
  Nulos        :   14,794  (22.4 %)
  Duplicados   :        0  (0.0 %)

  Columnas con nulos:
    referee                                    9.4 %
    venue_id                                  31.3 %
    venue_city                                 6.3 %
    home_win                                  28.2 %
    away_win                                  28.2 %
    home_goals                                 2.2 %
    away_goals                                 2.2 %
    home_goals_half_time                       2.2 %
    away_goals_half_time                       2.2 %
    home_goals_fulltime                        2.2 %
    away_goals_fulltime                        2.2 %
    home_goals_extra_time                     99.4 %
    away_goals_extratime                      99.4 %
    home_goals_penalty                        99.5 %
    away_goals_penalty                  